In [1]:
import os, sys, gc
import keras.backend as K

K.clear_session(); gc.collect()

WORK_DIR = '/kaggle/working/amr-5-class'
DATASET  = '/kaggle/input/datasets/gustavopolicarpo/rml201610a-dict/RML2016.10a_dict.dat'

import subprocess
if os.path.exists(WORK_DIR):
    subprocess.run(['git', '-C', WORK_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone',
                    'https://github.com/akshlabh/amr-5-class.git',
                    WORK_DIR], check=True)

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print("Working dir:", os.getcwd())

required = [
    'src/models/mcldnn_lstm1.py',
    'src/models/mcldnn_lstm64.py',
    'configs/exp_5class_lstm1.yaml',
    'configs/exp_5class_lstm64.yaml',
    'src/train_variants.py',
]
for f in required:
    print(f"  {'✓' if os.path.exists(f) else '✗ MISSING'}  {f}")

2026-06-23 19:40:15.771178: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782243615.951577      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782243616.007538      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782243616.447264      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782243616.447306      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782243616.447308      58 computation_placer.cc:177] computation placer alr

Working dir: /kaggle/working/amr-5-class
  ✓  src/models/mcldnn_lstm1.py
  ✓  src/models/mcldnn_lstm64.py
  ✓  configs/exp_5class_lstm1.yaml
  ✓  configs/exp_5class_lstm64.yaml
  ✗ MISSING  src/train_variants.py


In [2]:
os.environ['KERAS_BACKEND'] = 'tensorflow'
import keras

from src.models.mcldnn_lstm1 import MCLDNN_LSTM1
from src.models.mcldnn_lstm64 import MCLDNN_LSTM64
from src.models.mcldnn import MCLDNN

m_base   = MCLDNN(classes=5)
m_lstm1  = MCLDNN_LSTM1(classes=5)
m_lstm64 = MCLDNN_LSTM64(classes=5)

print(f"{'Model':<20} {'Params':>12}")
print(f"{'-'*34}")
print(f"{'Baseline (2xLSTM-128)':<20} {m_base.count_params():>12,}")
print(f"{'LSTM-1 (1xLSTM-128)':<20} {m_lstm1.count_params():>12,}")
print(f"{'LSTM-64 (2xLSTM-64)':<20} {m_lstm64.count_params():>12,}")

del m_base, m_lstm1, m_lstm64
K.clear_session(); gc.collect()

I0000 00:00:1782243660.532254      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model                      Params
----------------------------------
Baseline (2xLSTM-128)      404,401
LSTM-1 (1xLSTM-128)       272,817
LSTM-64 (2xLSTM-64)       222,641


0

In [4]:
print("="*60)
print("  Training MCLDNN-LSTM1 (single LSTM layer)")
print("="*60)

# train.py now supports lstm1 model type directly
!python src/train.py \
    --config configs/exp_5class_lstm1.yaml \
    --datasetpath {DATASET}

w = 'experiments/5class_lstm1/checkpoints/best_model.weights.h5'
print(f"\n{'✓' if os.path.exists(w) else '✗ MISSING'}  {w}")


  Training MCLDNN-LSTM1 (single LSTM layer)
[WARN] src/train_variants.py not found — using in-process fallback
[seed] All seeds fixed to 2016  (Python, NumPy, TensorFlow 2.19.0)
[dataset] Loading 5 classes: ['BPSK', 'QPSK', '8PSK', 'QAM16', 'QAM64']
[dataset] SNR range: -20 dB to 18 dB  (20 levels)
[dataset] After normalization — train RMS: 1.0000  (should be ≈1.0)
[dataset] Split: 60000 train | 20000 val | 20000 test samples


Model: "MCLDNN_LSTM1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input2 (InputLayer) │ (None, 128, 1)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input3 (InputLayer) │ (None, 128, 1)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_2 (Conv1D)    │ (None, 128, 50)   │        450 │ input2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_3 (Conv1D)    │ (None, 128, 50)   │        450 │ input3[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_i (Reshape) │ (None, 1, 128,    │          0 │ conv1_2[0][0]     │
│                     │ 50)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_q (Reshape) │ (None, 1, 128,    │          0 │ conv1_3[0][0]     │
│                     │ 50)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input1 (InputLayer) │ (None, 2, 128, 1) │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 2, 128,    │          0 │ reshape_i[0][0],  │
│ (Concatenate)       │ 50)               │            │ reshape_q[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_1 (Conv2D)    │ (None, 2, 128,    │        850 │ input1[0][0]      │
│                     │ 50)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2 (Conv2D)      │ (None, 2, 128,    │     20,050 │ concatenate[0][0] │
│                     │ 50)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 2, 128,    │          0 │ conv1_1[0][0],    │
│ (Concatenate)       │ 100)              │            │ conv2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv4 (Conv2D)      │ (None, 1, 124,    │    100,100 │ concatenate_1[0]… │
│                     │ 100)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_lstm        │ (None, 124, 100)  │          0 │ conv4[0][0]       │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 128)       │    117,248 │ reshape_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc1 (Dense)         │ (None, 128)       │     16,512 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop1 (Dropout)     │ (None, 128)       │          0 │ fc1[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc2 (Dense)         │ (None, 128)       │     16,512 │ drop1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop2 (Dropout)     │ (None, 128)       │          0 │ fc2[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ softmax (Dense)     │ (None, 5)         │        645 │ drop2[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 272,817 (1.04 MB)

 Trainable params: 272,817 (1.04 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10000


KeyboardInterrupt: 

In [ ]:
print("="*60)
print("  Training MCLDNN-LSTM64 (hidden size 64)")
print("="*60)

K.clear_session(); gc.collect()

# train.py now supports lstm64 model type directly
!python src/train.py \
    --config configs/exp_5class_lstm64.yaml \
    --datasetpath {DATASET}

w = 'experiments/5class_lstm64/checkpoints/best_model.weights.h5'
print(f"\n{'✓' if os.path.exists(w) else '✗ MISSING'}  {w}")


In [ ]:
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image, display

K.clear_session(); gc.collect()

from src.dataset import load_data, FIVE_CLASS
from src.models.mcldnn import MCLDNN
from src.models.mcldnn_lstm1 import MCLDNN_LSTM1
from src.models.mcldnn_lstm64 import MCLDNN_LSTM64
from src.models.mcldnn_attention import build_mcldnn_attention

(mods, snrs, lbl), _, _, (X_test, Y_test), (_, _, test_idx) = load_data(
    DATASET, FIVE_CLASS, seed=2016
)
test_SNRs = np.array([lbl[test_idx[j]][1] for j in range(len(test_idx))])
ALL_SNRS  = sorted(set(test_SNRs))
print(f"Test set: {len(test_idx)} samples across {len(ALL_SNRS)} SNRs")

def make_inputs(X):
    return [np.expand_dims(X, 3).astype('float32'),
            np.expand_dims(X[:,0,:], 2).astype('float32'),
            np.expand_dims(X[:,1,:], 2).astype('float32')]
inp_test = make_inputs(X_test)

model_configs = {
    'Baseline (2xLSTM-128, 404k)': {
        'weights': 'experiments/5class_baseline/checkpoints/best_model.weights.h5',
        'build'  : lambda: MCLDNN(classes=5),
        'color': '#1f77b4', 'marker': 'o', 'ls': '-',
    },
    'Attention (193k)': {
        'weights': 'experiments/5class_attention/checkpoints/best_model.weights.h5',
        'build'  : lambda: build_mcldnn_attention(classes=5),
        'color': '#d62728', 'marker': '^', 'ls': '-.',
    },
    'LSTM-1 (273k)': {
        'weights': 'experiments/5class_lstm1/checkpoints/best_model.weights.h5',
        'build'  : lambda: MCLDNN_LSTM1(classes=5),
        'color': '#2ca02c', 'marker': 's', 'ls': '--',
    },
    'LSTM-64 (223k)': {
        'weights': 'experiments/5class_lstm64/checkpoints/best_model.weights.h5',
        'build'  : lambda: MCLDNN_LSTM64(classes=5),
        'color': '#9467bd', 'marker': 'D', 'ls': ':',
    },
}

model_accs = {}
for name, cfg in model_configs.items():
    if not os.path.exists(cfg['weights']):
        print(f"[SKIP] {name} — weights not found: {cfg['weights']}")
        continue
    K.clear_session(); gc.collect()
    m = cfg['build']()
    m.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    m.load_weights(cfg['weights'])
    accs = {}
    for snr in ALL_SNRS:
        mk = test_SNRs == snr
        accs[snr] = m.evaluate([b[mk] for b in inp_test], Y_test[mk],
                                verbose=0, batch_size=400)[1]
    model_accs[name] = {'accs': accs, 'params': m.count_params(),
                        'color': cfg['color'], 'marker': cfg['marker'], 'ls': cfg['ls']}
    print(f"✓ {name} — {m.count_params():,} params — "
          f"mean acc: {np.mean(list(accs.values()))*100:.2f}%")

# ── Print full table ───────────────────────────────────────────────────────────
print(f"\n{'='*100}")
print(f"  FOUR-WAY COMPARISON — All 20 SNRs, same test set, seed=2016")
print(f"{'='*100}")
names = list(model_accs.keys())
print(f"  {'SNR':>6}", end='')
for n in names: print(f"  {n:>22}", end='')
print()
for snr in ALL_SNRS:
    print(f"  {snr:>5} dB", end='')
    for n in names:
        print(f"  {model_accs[n]['accs'][snr]*100:>21.2f}%", end='')
    print()
print(f"  {'-'*94}")
print(f"  {'Mean':>6}", end='')
for n in names:
    print(f"  {np.mean(list(model_accs[n]['accs'].values()))*100:>21.2f}%", end='')
print()
print(f"  {'Params':>6}", end='')
for n in names:
    print(f"  {model_accs[n]['params']:>21,}", end='')
print()
print(f"{'='*100}")

os.makedirs('experiments/comparison_lstm_variants', exist_ok=True)
df_all = pd.DataFrame({'snr': ALL_SNRS,
                       **{n.split(' (')[0]: [model_accs[n]['accs'][s] for s in ALL_SNRS]
                          for n in names}})
df_all.to_csv('experiments/comparison_lstm_variants/fourway_comparison.csv', index=False)
print(f"\n[SAVED] experiments/comparison_lstm_variants/fourway_comparison.csv")

In [ ]:
# ── Plot 1: All 4 models, accuracy vs SNR ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 7))
for name, r in model_accs.items():
    ys = [r['accs'][s]*100 for s in ALL_SNRS]
    ax.plot(ALL_SNRS, ys, marker=r['marker'], color=r['color'], ls=r['ls'],
            linewidth=2.3, markersize=8, label=f"{name}")
ax.set_xlabel('SNR (dB)', fontsize=13)
ax.set_ylabel('Test Accuracy (%)', fontsize=13)
ax.set_title('Four-Way Comparison: Baseline vs Attention vs LSTM-1 vs LSTM-64\n'
             'Same test set, same seed=2016, all 20 SNRs',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_xticks(ALL_SNRS)
plt.xticks(rotation=45)
plt.tight_layout()
p1 = 'experiments/comparison_lstm_variants/fourway_acc_vs_snr.png'
fig.savefig(p1, dpi=150, bbox_inches='tight')
plt.close()
display(Image(p1))
print(f"[SAVED] {p1}")

# ── Plot 2: Params vs Mean Accuracy (efficiency scatter) ─────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
for name, r in model_accs.items():
    mean_acc = np.mean(list(r['accs'].values())) * 100
    ax.scatter(r['params'], mean_acc, s=200, color=r['color'],
              marker=r['marker'], edgecolor='black', linewidth=1.2, zorder=3)
    ax.annotate(name.split(' (')[0], (r['params'], mean_acc),
               textcoords='offset points', xytext=(8, 5), fontsize=10, fontweight='bold')
ax.set_xlabel('Total Parameters', fontsize=12)
ax.set_ylabel('Mean Accuracy (%)', fontsize=12)
ax.set_title('Parameter Efficiency: Accuracy vs Model Size', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
p2 = 'experiments/comparison_lstm_variants/param_efficiency.png'
fig.savefig(p2, dpi=150, bbox_inches='tight')
plt.close()
display(Image(p2))
print(f"[SAVED] {p2}")

# ── Plot 3 & 4: training curves for LSTM-1 and LSTM-64 ────────────────────────
for tag, logdir in [('LSTM-1', 'experiments/5class_lstm1/logs'),
                    ('LSTM-64', 'experiments/5class_lstm64/logs')]:
    log_path = f'{logdir}/training_log.csv'
    if not os.path.exists(log_path):
        print(f"[SKIP] {tag} log not found")
        continue
    log = pd.read_csv(log_path)
    acc_col     = next((c for c in log.columns if 'accuracy' in c and 'val' not in c), 'accuracy')
    val_acc_col = next((c for c in log.columns if 'val' in c and 'accuracy' in c), 'val_accuracy')
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f'{tag} Training History', fontsize=13, fontweight='bold')
    axes[0].plot(log['epoch'], log['loss'], color='#1f77b4', label='Train loss')
    axes[0].plot(log['epoch'], log['val_loss'], color='#d62728', label='Val loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(log['epoch'], log[acc_col]*100, color='#1f77b4', label='Train acc')
    axes[1].plot(log['epoch'], log[val_acc_col]*100, color='#d62728', label='Val acc')
    axes[1].axhline(20, color='gray', linestyle='--', alpha=0.6, label='Random')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    p = f'experiments/comparison_lstm_variants/{tag.lower().replace("-","")}_training_history.png'
    fig.savefig(p, dpi=150, bbox_inches='tight')
    plt.close()
    display(Image(p))
    print(f"[SAVED] {p}")
    best = log['val_accuracy'].idxmax() if 'val_accuracy' in log.columns else log[val_acc_col].idxmax()
    print(f"  Best epoch: {log.loc[best,'epoch']}  val_acc={log.loc[best,val_acc_col]*100:.2f}%")

# ── Plot 5 & 6: confusion matrices for LSTM-1 and LSTM-64 ─────────────────────
for tag, figdir in [('LSTM-1', 'experiments/5class_lstm1/figures'),
                    ('LSTM-64', 'experiments/5class_lstm64/figures')]:
    cm_path = f'{figdir}/confusion_all_snrs.png'
    if os.path.exists(cm_path):
        print(f"\n{tag} confusion matrix:")
        display(Image(cm_path))

In [ ]:
import zipfile, glob
from datetime import datetime

ts      = datetime.now().strftime('%Y%m%d_%H%M')
zip_out = f'/kaggle/working/all_variants_results_{ts}.zip'

include_dirs = [
    'experiments/5class_baseline',
    'experiments/5class_attention',
    'experiments/5class_lstm1',
    'experiments/5class_lstm64',
    'experiments/comparison_lstm_variants',
]
exts = ('.png', '.csv', '.h5', '.txt', '.yaml')

with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for d in include_dirs:
        for f in glob.glob(f'{d}/**/*', recursive=True):
            if os.path.isfile(f) and f.endswith(exts):
                zf.write(f)
                print(f"  + {f}")

size = os.path.getsize(zip_out)/1e6
print(f"\n[DONE] {zip_out}  ({size:.1f} MB)")
print("Download from Kaggle Output panel →")